In [1]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

In [2]:
ELASTICSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "ev-vehicles"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [3]:
es_client = Elasticsearch(ELASTICSEARCH_URL)

print("Elasticsearch connected:", es_client.info())

Elasticsearch connected: {'name': '3016a6d3053d', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Ex3VVj5uSn-tVMlx6zVcpg', 'version': {'number': '8.15.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179', 'build_date': '2024-08-05T10:05:34.233336849Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
# Check the number of documents:

document_count = es_client.count(index=INDEX_NAME)["count"]

print(f"BEV documents in Elasticsearch: {document_count:,}")

BEV documents in Elasticsearch: 1,572


In [5]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [6]:
SOURCE_FIELDS = [
    "id",
    "year",
    "make",
    "model",
    "vehicle_name",
    "vehicle_class",
    "drive",
    "electric_range_miles",
    "city_mpge",
    "highway_mpge",
    "combined_mpge",
    "charge_120v_hours",
    "charge_240v_hours",
    "ev_motor",
    "annual_fuel_cost_usd",
    "document_text",
]

In [7]:
def show_results(results):
    """Print retrieved BEV records in a readable format."""

    for rank, result in enumerate(results, start=1):
        source = result["_source"]

        print(f"Rank {rank}")
        print(f"Vehicle: {source.get('vehicle_name')}")
        print(f"FuelEconomy.gov ID: {source.get('id')}")
        print(f"Electric range: {source.get('electric_range_miles')} miles")
        print(f"Combined efficiency: {source.get('combined_mpge')} MPGe")
        print(f"Vehicle class: {source.get('vehicle_class')}")
        print(f"Score: {result.get('_score', 'N/A')}")
        print("-" * 80)

In [8]:
def text_search(query, number_of_results=5):
    """
    Search for BEVs using keyword and full-text matching.
    """

    response = es_client.search(
        index=INDEX_NAME,
        size=number_of_results,
        source=SOURCE_FIELDS,
        query={
            "multi_match": {
                "query": query,
                "fields": [
                    "vehicle_name^4",
                    "make^3",
                    "model^3",
                    "vehicle_class^2",
                    "document_text",
                ],
                "fuzziness": "AUTO",
            }
        },
    )

    return response["hits"]["hits"]

In [9]:
query = "Tesla Model 3"

text_results = text_search(query)

show_results(text_results)

Rank 1
Vehicle: 2022 Tesla Model 3 RWD
FuelEconomy.gov ID: 45013
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2023 Tesla Model 3 RWD
FuelEconomy.gov ID: 46206
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Tesla Model 3 RWD
FuelEconomy.gov ID: 47909
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2026 Tesla Model 3 Performance
FuelEconomy.gov ID: 50036
Electric range: 309 miles
Combined efficiency: 114 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank

In [10]:
query = "electric SUV with long range"

text_results = text_search(query)

show_results(text_results)

Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 33.769302
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 48359
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2025 Tesla Cybertruck Long Range
FuelEconomy.gov ID: 49152
Electric range: 335 miles
Combined efficiency: 82 MPGe
Vehicle class: Stan

In [11]:
def vector_search(query, number_of_results=5):
    """
    Search for BEVs using sentence-transformer vector similarity.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True,
    ).tolist()

    response = es_client.search(
        index=INDEX_NAME,
        size=number_of_results,
        source=SOURCE_FIELDS,
        knn={
            "field": "embedding",
            "query_vector": query_embedding,
            "k": number_of_results,
            "num_candidates": 100,
        },
    )

    return response["hits"]["hits"]

In [12]:
query = "Which fully electric cars can travel the farthest before recharging?"

vector_results = vector_search(query)

show_results(vector_results)

Rank 1
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Goodyear
FuelEconomy.gov ID: 48783
Electric range: 268 miles
Combined efficiency: 85 MPGe
Vehicle class: Large Cars
Score: 0.77642226
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Nexen
FuelEconomy.gov ID: 48784
Electric range: 308 miles
Combined efficiency: 98 MPGe
Vehicle class: Large Cars
Score: 0.77581716
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Mercedes-Benz EQE 500 4matic (SUV)
FuelEconomy.gov ID: 48391
Electric range: 264 miles
Combined efficiency: 81 MPGe
Vehicle class: Midsize Station Wagons
Score: 0.7733977
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 18in
FuelEconomy.gov ID: 48782
Electric range: 274 miles
Combined efficiency: 87 MPGe
Vehicle class: Large Cars
Score: 0.7

In [13]:
query = "Show efficient electric cars that use little energy"

vector_results = vector_search(query)

show_results(vector_results)

Rank 1
Vehicle: 2026 Porsche Macan 4S Electric
FuelEconomy.gov ID: 50295
Electric range: 290 miles
Combined efficiency: 92 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.77103305
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Lexus RZ 450e AWD (20 inch wheels - 235/50R20,255/45R20)
FuelEconomy.gov ID: 50219
Electric range: 228 miles
Combined efficiency: 95 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.769799
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2027 Mercedes-Benz EQS 400 4matic (SUV)
FuelEconomy.gov ID: 50662
Electric range: 312 miles
Combined efficiency: 79 MPGe
Vehicle class: Standard Sport Utility Vehicle 4WD
Score: 0.76859665
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2013 Fiat 500e
FuelEconomy.gov ID: 33396
Electric range: 87 miles
Combined efficiency: 116 MPGe
Vehicle class: Minicompac

# Compare text and vector results

In [14]:
query = "Which electric cars can drive the farthest without charging?"

In [15]:
print("TEXT SEARCH RESULTS")
print("=" * 80)

text_results = text_search(query)
show_results(text_results)

TEXT SEARCH RESULTS
Rank 1
Vehicle: 2011 smart fortwo electric drive cabriolet
FuelEconomy.gov ID: 31064
Electric range: 63 miles
Combined efficiency: 87 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2011 smart fortwo electric drive coupe
FuelEconomy.gov ID: 31065
Electric range: 63 miles
Combined efficiency: 87 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2013 smart fortwo electric drive convertible
FuelEconomy.gov ID: 33305
Electric range: 68 miles
Combined efficiency: 107 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2013 smart fortwo electric drive coupe
FuelEconomy.gov ID: 33306
Electric range: 68 miles
Combined efficiency: 107 MPGe
Vehicle class: Two Seaters
Score: 30.487679
---------

In [16]:
print("VECTOR SEARCH RESULTS")
print("=" * 80)

vector_results = vector_search(query)
show_results(vector_results)

VECTOR SEARCH RESULTS
Rank 1
Vehicle: 2025 Tesla Model 3 Long Range RWD-I (19in wheels)
FuelEconomy.gov ID: 49154
Electric range: 346 miles
Combined efficiency: 131 MPGe
Vehicle class: Midsize Cars
Score: 0.7784989
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Tesla Model 3 Premium RWD
FuelEconomy.gov ID: 50038
Electric range: 363 miles
Combined efficiency: 137 MPGe
Vehicle class: Midsize Cars
Score: 0.7773638
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Tesla Model 3 Long Range RWD
FuelEconomy.gov ID: 48795
Electric range: 363 miles
Combined efficiency: 137 MPGe
Vehicle class: Midsize Cars
Score: 0.77476215
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Nexen
FuelEconomy.gov ID: 48784
Electric range: 308 miles
Combined efficiency: 98 MPGe
Vehicle class: Large Cars
Score: 0.7746658

# Build hybrid search with Reciprocal Rank Fusion

In [ ]:
# Build hybrid search with Reciprocal Rank Fusion


def hybrid_search(
    query,
    number_of_results=5,
    candidate_results=20,
    rrf_constant=60,
):
    """
    Combine text and vector results using Reciprocal Rank Fusion.
    """

    text_results = text_search(
        query,
        number_of_results=candidate_results,
    )

    vector_results = vector_search(
        query,
        number_of_results=candidate_results,
    )

    combined_results = {}

    # Add scores based on text-search ranking
    for rank, result in enumerate(text_results, start=1):
        document_id = result["_id"]

        if document_id not in combined_results:
            combined_results[document_id] = {
                "_source": result["_source"],
                "_score": 0,
                "text_rank": None,
                "vector_rank": None,
            }

        combined_results[document_id]["_score"] += 1 / (
            rrf_constant + rank
        )

        combined_results[document_id]["text_rank"] = rank

    # Add scores based on vector-search ranking
    for rank, result in enumerate(vector_results, start=1):
        document_id = result["_id"]

        if document_id not in combined_results:
            combined_results[document_id] = {
                "_source": result["_source"],
                "_score": 0,
                "text_rank": None,
                "vector_rank": None,
            }

        combined_results[document_id]["_score"] += 1 / (
            rrf_constant + rank
        )

        combined_results[document_id]["vector_rank"] = rank

    ranked_results = sorted(
        combined_results.values(),
        key=lambda result: result["_score"],
        reverse=True,
    )

    return ranked_results[:number_of_results]

In [18]:
query = "Which battery electric vehicles have the longest driving range?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2020 Tesla Model S Long Range
FuelEconomy.gov ID: 42282
Electric range: 373 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.016129032258064516
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2019 Tesla Model S Long Range
FuelEconomy.gov ID: 41417
Electric range: 370 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Ca

In [19]:
query = "What are the specifications of the Tesla Model Y?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 2022 Tesla Model Y RWD
FuelEconomy.gov ID: 45017
Electric range: 244 miles
Combined efficiency: 129 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.032266458495966696
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Tesla Model Y RWD
FuelEconomy.gov ID: 48476
Electric range: 260 miles
Combined efficiency: 120 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.031754032258064516
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2020 Tesla Model Y Performance AWD
FuelEconomy.gov ID: 42474
Electric range: 315 miles
Combined efficiency: 121 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.029877369007803793
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2023 Tesla Model Y Performance AWD
FuelEconomy.gov ID: 46213
Electric range: 303 miles
Combined efficiency: 111 MPGe
Vehicle class: Small Spo

In [20]:
query = "Which electric cars have high MPGe efficiency?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30976
Electric range: 33 miles
Combined efficiency: 55 MPGe
Vehicle class: Small Pickup Trucks 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Lucid Air Pure RWD with 19 inch wheels
FuelEconomy.gov ID: 49969
Electric range: 420 miles
Combined efficiency: 146 MPGe
Vehicle class: Large Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30977
Electric range: 72 miles
Combined efficiency: 28 MPGe
Vehicle class: Small Pickup Trucks 2WD
Score: 0.016129032258064516
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2027 Mercedes-Benz EQE 320 Plus (SUV)
FuelEconomy.gov ID: 50661
Electric range: 302 miles
Combined efficiency: 93 MPGe
Vehicle class: Midsize Station Wagons
Score: 0

In [ ]:

"""
Text-search rank: 1
Vector-search rank: 2

This means the vehicle was considered relevant by both retrieval approaches, which is generally a strong sign.
"""



query = "electric SUV with long range"

hybrid_results = hybrid_search(query)

for rank, result in enumerate(hybrid_results, start=1):
    source = result["_source"]

    print(f"Final hybrid rank: {rank}")
    print(f"Vehicle: {source['vehicle_name']}")
    print(f"RRF score: {result['_score']:.4f}")
    print(f"Text-search rank: {result['text_rank']}")
    print(f"Vector-search rank: {result['vector_rank']}")
    print("-" * 80)

NameError: name 'hybrid_search' is not defined

In [22]:
test_questions = [
    "What is the range of the Tesla Model 3?",
    "Which battery electric vehicles have the longest driving range?",
    "Show electric SUVs with good range.",
    "Which fully electric cars are the most efficient?",
    "What EVs have the shortest Level 2 charging time?",
    "Show electric vehicles made by Nissan.",
    "Which electric cars have all-wheel drive?",
    "What electric cars are in the small SUV class?",
]

In [23]:
query = test_questions[2]

print("QUESTION:")
print(query)

print("\nTEXT SEARCH")
show_results(text_search(query))

print("\nVECTOR SEARCH")
show_results(vector_search(query))

print("\nHYBRID SEARCH")
show_results(hybrid_search(query))

QUESTION:
Show electric SUVs with good range.

TEXT SEARCH
Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 48359
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 4
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30976
Electric range: 33 m